In [2]:
import json
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download the 'wordnet' and 'stopwords' resources from NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Load intents
with open('/content/intents.json') as f:
    data = json.load(f)

df = pd.DataFrame(data['intents'])

# Expand patterns
rows = []
for _, row in df.iterrows():
    for pattern in row['patterns']:
        rows.append({
            'tag': row['tag'],
            'pattern': pattern,
            'responses': row['responses']
        })

df = pd.DataFrame(rows)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
import nltk
nltk.download('punkt_tab')

lemmatizer = WordNetLemmatizer() #Converts words to their base form
stop_words = set(stopwords.words('english'))#Removes common words (e.g., is, the, and) that do not contribute to intent detection

def process_sentence(sentence):
    words = nltk.word_tokenize(sentence.lower())#Converts all input text to lowercase to ensure consistency
    return " ".join([
        lemmatizer.lemmatize(word)
        for word in words
        if word.isalpha() and word not in stop_words
    ])

df['pattern'] = df['pattern'].apply(process_sentence)


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder

tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['pattern'])

X = tokenizer.texts_to_sequences(df['pattern'])
X = pad_sequences(X, padding='post')

lbl_enc = LabelEncoder()
y = lbl_enc.fit_transform(df['tag'])


In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(len(tokenizer.word_index) + 1, 100, input_length=X.shape[1]),
    LSTM(64),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(len(np.unique(y)), activation='softmax')
])

model.compile(
    optimizer='adam', # adam = gradient descent method
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)



In [6]:
model.fit(X, y, epochs=50, batch_size=8, verbose=1)
#Trains chatbot for 50 epochs

Epoch 1/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.1052 - loss: 2.7733
Epoch 2/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1528 - loss: 2.7621     
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1762 - loss: 2.7519 
Epoch 4/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1715 - loss: 2.7272 
Epoch 5/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1370 - loss: 2.7040 
Epoch 6/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1956 - loss: 2.6477 
Epoch 7/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1516 - loss: 2.5674     
Epoch 8/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2243 - loss: 2.3660 
Epoch 9/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2355 - loss: 2.2556 
Epoch 10/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4940 - loss: 1.9352 
Epoch 11/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4646 - loss: 1.9050
Epoch 12/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - acc

In [7]:
CONFIDENCE_THRESHOLD = 0.65
import random


In [8]:
def model_response(query):
    processed = process_sentence(query)

    seq = tokenizer.texts_to_sequences([processed])
    padded = pad_sequences(seq, maxlen=X.shape[1], padding='post')

    probs = model.predict(padded)[0]
    confidence = np.max(probs)
    idx = np.argmax(probs)

    if confidence < CONFIDENCE_THRESHOLD:
        print("Chatbot: I'm not confident about that. Please rephrase your question or contact the help desk.")
        return

    tag = lbl_enc.inverse_transform([idx])[0]
    responses = df[df['tag'] == tag]['responses'].values[0]

    print(f"Chatbot (confidence {confidence:.2f}): {random.choice(responses)}")


In [10]:
print("Chatbot: Welcome to the University Student FAQ Bot 🎓")
print("Chatbot: Ask me about admissions, courses, fees, exams, hostels, or placements.")
print("Type 'quit' to exit.\n")

while True:
    message = input("You: ").strip()

    if message.lower() == "quit":
        print("Chatbot: Thank you for using the University FAQ Bot. Goodbye!")
        break

    if not message:
        print("Chatbot: Please enter a valid question.")
        continue

    model_response(message)


Chatbot: Welcome to the University Student FAQ Bot 🎓
Chatbot: Ask me about admissions, courses, fees, exams, hostels, or placements.
Type 'quit' to exit.

You: Hi!
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Chatbot (confidence 1.00): Hi! Ask me anything about the university.
You: How is the student admission process?
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Chatbot (confidence 0.95): You can apply online through the university admission portal by submitting the required documents.
You: What courses available in this university?
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Chatbot (confidence 1.00): The university offers undergraduate, postgraduate, and doctoral programs across various disciplines.
You: How about fees for the study?
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
Chatbot (confidence 1.00): Tuition fees vary by program. Detailed fee structures are available on the university website.
You: Students got their hostel in the campus?
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Chatbot (confidence 0.98): Ye